In [5]:
#importing files.
from google.colab import files
uploaded = files.upload()

Saving archive (1).zip to archive (1).zip


In [10]:
#Unzipping the file
import zipfile

with zipfile.ZipFile('archive (1).zip', 'r') as zip_ref:
    zip_ref.extractall('tickets_folder')

In [11]:
#for Seeing the files inside
import os
os.listdir('tickets_folder')

['taxis_cleaned.csv']

In [12]:
#installing the Libraries
!pip install nltk spacy scikit-learn pandas

In [13]:
#importing Libraries
import pandas as pd
import numpy as np
import nltk
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [14]:
#Downloading stopwords:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [16]:
#Upload dataset to Colab.
df = pd.read_csv("tickets_folder/taxis_cleaned.csv")
df.head()

,pickup,dropoff,passengers,distance,fare,tip,tolls,total,color,payment,pickup_zone,dropoff_zone,pickup_borough,dropoff_borough
0,2019-03-23 20:21:09,2019-03-23 20:27:24,1,1.60,7.0,2.15,0.0,12.95,yellow,credit card,Lenox Hill West,UN/Turtle Bay South,Manhattan,Manhattan
1,2019-03-04 16:11:55,2019-03-04 16:19:00,1,0.79,5.0,0.00,0.0,9.30,yellow,cash,Upper West Side South,Upper West Side South,Manhattan,Manhattan
2,2019-03-27 17:53:01,2019-03-27 18:00:25,1,1.37,7.5,2.36,0.0,14.16,yellow,credit card,Alphabet City,West Village,Manhattan,Manhattan
3,2019-03-10 01:23:59,2019-03-10 01:49:51,1,7.70,27.0,6.15,0.0,36.95,yellow,credit card,Hudson Sq,Yorkville West,Manhattan,Manhattan
4,2019-03-30 13:27:42,2019-03-30 13:37:14,3,2.16,9.0,1.10,0.0,13.40,yellow,credit card,Midtown East,Yorkville West,Manhattan,Manhattan


In [17]:
#Check Dataset Structure
df.columns

Index(['pickup', 'dropoff', 'passengers', 'distance', 'fare', 'tip', 'tolls',
       'total', 'color', 'payment', 'pickup_zone', 'dropoff_zone',
       'pickup_borough', 'dropoff_borough'],
      dtype='object')

In [19]:
#Combine text from relevant columns
df['ticket_text'] = df['pickup_zone'] + " " + df['dropoff_zone'] + " " + df['pickup_borough'] + " " + df['dropoff_borough']
df[['ticket_text','total','color']].head()

,ticket_text,total,color
0,Lenox Hill West UN/Turtle Bay South Manhattan ...,12.95,yellow
1,Upper West Side South Upper West Side South Ma...,9.30,yellow
2,Alphabet City West Village Manhattan Manhattan,14.16,yellow
3,Hudson Sq Yorkville West Manhattan Manhattan,36.95,yellow
4,Midtown East Yorkville West Manhattan Manhattan,13.40,yellow


In [20]:
#Text Cleaning
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)

stop_words = set(stopwords.words('english'))

def clean_text(text):

    text = str(text).lower()
    text = re.sub(r'[^\u017F\w\s]', '', text)

    words = text.split()
    words = [word for word in words if word not in stop_words]

    return " ".join(words)

df['clean_text'] = df['ticket_text'].apply(clean_text)

In [22]:
#Convert Text to TF-IDF Features(Feature Extraction)
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['clean_text'])
y = df['color']

In [23]:
#Train Model
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [24]:
#Model Evaluation
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

Accuracy: 0.9453551912568307
              precision    recall  f1-score   support

       green       0.82      0.83      0.82       199
      yellow       0.97      0.97      0.97      1082

    accuracy                           0.95      1281
   macro avg       0.89      0.90      0.90      1281
weighted avg       0.95      0.95      0.95      1281



In [25]:
#Predict Priority
ticket = ["My account login is not working"]

ticket_vec = vectorizer.transform(ticket)

predicted_category = model.predict(ticket_vec)[0]

print("Predicted Category:", predicted_category)

Predicted Category: yellow


In [27]:
df['color'].unique()

array(['yellow', 'green'], dtype=object)

In [28]:
#Adding Confusion Matrix
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

[[ 165   34]
 [  36 1046]]


In [29]:
import pickle

# after training model
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

In [31]:
# Install Streamlit
!pip install streamlit

import streamlit as st
import pickle
import numpy as np

# Title
st.title("🚖 Taxi Fare Prediction App")

# Load model
with open("model.pkl", "rb") as f:
    model = pickle.load(f)

st.header("Enter Trip Details")

# Inputs (change based on your dataset)
distance = st.number_input("Distance (km)", min_value=0.0)
passengers = st.number_input("Passengers", min_value=1)

# Predict button
if st.button("Predict Fare"):
    input_data = np.array([[distance, passengers]])
    prediction = model.predict(input_data)

    st.success(f"Estimated Fare: ₹{prediction[0]:.2f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 107.5 MB/s eta 0:00:00


2026-03-20 17:08:36.293 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-20 17:08:36.399 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-03-20 17:08:36.399 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-20 17:08:36.400 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-20 17:08:36.402 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-20 17:08:36.403 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-20 17:08:36.405 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-20 17:08:36.410 Thread 'MainThread': mi

In [34]:
%%writefile app.py
import streamlit as st
import pickle
import numpy as np

st.title("🚖 Taxi Fare Prediction App")

with open("model.pkl", "rb") as f:
    model = pickle.load(f)

st.header("Enter Trip Details")
distance = st.number_input("Distance (km)", min_value=0.0)
passengers = st.number_input("Passengers", min_value=1)

if st.button("Predict Fare"):
    st.warning("This app was designed for fare prediction, but the trained model predicts taxi 'color' based on text features. Adjusting input for current model...")

    text_input = st.text_input("Enter pickup zone, dropoff zone, pickup borough, dropoff borough (e.g., 'Lenox Hill West UN/Turtle Bay South Manhattan Manhattan')")

    if text_input and st.button("Predict Taxi Color"):

        clean_text_input = clean_text(text_input)
        input_vec = vectorizer.transform([clean_text_input])
        predicted_color = model.predict(input_vec)[0]
        st.success(f"Predicted Taxi Color: {predicted_color}")


Overwriting app.py


In [35]:
from google.colab import files
files.download("app.py")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:
from google.colab import files
files.download("model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>